# Comparing Optimizers

This notebook will compare 3 different optimizers when used on the MNIST feedforward neural network

## Neural Network and Data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

class FeedForwardNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        x = torch.relu(x)
        x = self.fc3(x)
        return x


## Defining the Training Loop

In [ ]:
def train_model(model, optimizer, criterion, train_loader, epochs):

    losses = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0 

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)
            running_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        average_loss = running_loss / len(train_loader)
        losses.append(average_loss)
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {average_loss:.4f}")  
    return losses

## Training Models on Different Optimizers

In [ ]:
criterion = nn.CrossEntropyLoss()

## Same random seed ensures all models start with the same initial weights for a fair comparison
torch.manual_seed(42)
sgd_model = FeedForwardNN().to(device)

torch.manual_seed(42)
momentum_model = FeedForwardNN().to(device)

torch.manual_seed(42)
adam_model = FeedForwardNN().to(device)

## SGD optimizer without momentum
optimizer_sgd = optim.SGD(sgd_model.parameters(), lr=0.01)
## SGD optimizer with momentum
optimizer_momentum = optim.SGD(momentum_model.parameters(), lr=0.01, momentum=0.9)
## Adam optimizer
optimizer_adam = optim.Adam(adam_model.parameters(), lr=0.001)

## Call the training function for each optimizer
sgd_losses = train_model(sgd_model, optimizer_sgd, criterion, train_loader, epochs=10)
momentum_losses = train_model(momentum_model, optimizer_momentum, criterion, train_loader, epochs=10)
adam_losses = train_model(adam_model, optimizer_adam, criterion, train_loader, epochs=10)

## Plotting the Loss Curves

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(sgd_losses, label="SGD")
plt.plot(momentum_losses, label="SGD + Momentum")
plt.plot(adam_losses, label="Adam")

plt.xlabel("Epoch")
plt.ylabel("Average Training Loss")
plt.title("Training Loss by Optimizer")
plt.legend()
plt.grid(True)

plt.show()

## Results

In this experiment, standard SGD converged the slowest because it updates the model using only the current gradient, which can lead to slower progress and oscillations. SGD with Momentum converged faster because it incorporates information from previous updates, allowing it to maintain movement in beneficial directions and reduce oscillations. Adam also converged rapidly by combining momentum with adaptive learning rates for each parameter. In this experiment, SGD with Momentum achieved the lowest final loss, showing that although Adam is often an excellent default optimizer, it does not always outperform SGD with Momentum on every dataset or model.